# Scikit-learn - Topic 6 - Cross Validation Search (GridSearchCV) and Hyperparameter Optimisation - Multiple Clf

## <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%202%20-%20Unit%20Objective.png"> Topic Objectives

* Learn and use GridSearchCV for Hyperparameter Optimisation




---

## <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%204%20-%20Import%20Package%20for%20Learning.png"> Import Packages for Learning

We will install scikit-learn, xgboost, feature-engine and yellow brick to run our exercises.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

---

## <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%2010-%20Lesson%20Content.png"> Scikit-learn - Topic 6 - Cross Validation Search (GridSearchCV) and Hyperparameter Optimisation

### <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%2010-%20Lesson%20Content.png">  Hyperparameter Optimisation with one algorithm

---

#### <img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%2010-%20Lesson%20Content.png">  Multiclass Classification

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> In the last section, we saw how to conduct hyperparameter tuning using one algorithm to solve a Binary Classification problem.
* There is a tiny difference in using GridSearchCV when your ML task is multi-class classification. We will cover that now.


<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png"> We are going to consider a similar workflow we studied earlier:
* Split the data
* Define the pipeline and hyperparameters
* Fit the pipeline
* Evaluate the pipeline

We load the iris dataset for this exercise. It contains records of three species or classes of iris plants, with their petal and sepal measurements.

In [2]:
df_clf = sns.load_dataset('iris')

print(df_clf.shape)
df_clf.head()

(150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


As usual, we split the data into train and test set.

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train, y_test = train_test_split(
                                    df_clf.drop(['species'],axis=1),
                                    df_clf['species'],
                                    test_size=0.2,
                                    random_state=101
                                    )

print("* Train set:", X_train.shape, y_train.shape, "\n* Test set:", X_test.shape, y_test.shape)

* Train set: (120, 4) (120,) 
* Test set: (30, 4) (30,)


And create a pipeline using three steps: feature scaling, feature selection and modelling with RandomForestClassifier.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

def pipeline_clf():
  pipeline = Pipeline([
      ( "feat_scaling",StandardScaler() ),
      ( "feat_selection",SelectFromModel(RandomForestClassifier(random_state=101)) ),
      ( "model", RandomForestClassifier(random_state=101)),

    ])

  return pipeline


<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%205%20-%20Practice.png"> We define our hyperparameter list based on the algorithm documentation.
* In this case, there will be two hyperparameter combinations.
* As the intention of the unit is to learn hyperparameter optimisation, we will reduce the number of hyperparameter combinations so the code runs faster. However, we encourage you to try additional larger combinations to consolidate your learning.



In [5]:
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
from sklearn.model_selection import GridSearchCV

param_grid = {"model__n_estimators":[10,20],
              }
param_grid

{'model__n_estimators': [10, 20]}

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%207-%20Note.png">  Let's assume for this project, the client is interested particularly in the Virginica species and needs the predictions for this class to be precise. Although this is an arbitrary choice, in this case, it is an example of the type of business requirements given to you by the product owner or business expert.  
* In this case, your scoring parameter is `precision_score` to the class Virginica.
  * In a Multiclass classification, when your performance metric is accuracy, you just pass scoring='accuracy' as an argument, as done with a binary classifier.
  * In our case, we need to pass arguments to the make_scorer() method to fine-tune the model using precision on the Virginica species. We pass to make_scorer as an argument the metric we want - `precision_score`. The next argument is `labels`, where you set the class you want to tune as a list. Note that in this dataset, the species is not encoded as numbers but as categories. If it were numbers, you would pass the number related to the class you want to tune. The last argument is `average`, and it should equal `None` since you compute the precision from one class only (in this case Virginica) and you don't need to average.
* Finally, you fit the grid search to the training data.


In [6]:
from sklearn.metrics import make_scorer, precision_score
grid = GridSearchCV(estimator=pipeline_clf(),
                    param_grid=param_grid,
                    cv=2,
                    n_jobs=-2,
                    verbose=3, # In the workplace we typically set verbose to 1, 
                    # to reduce the number of messages when fitting the models.
                    # For teaching purposes, we set it to 3 to see the score for each cross-validated model.
                    scoring=make_scorer(precision_score,
                                        labels=['virginica'],
                                        average=None)
                    )


grid.fit(X_train,y_train)

Fitting 2 folds for each of 2 candidates, totalling 4 fits
[CV 1/2] END ............model__n_estimators=10;, score=1.000 total time=   0.2s
[CV 2/2] END ............model__n_estimators=10;, score=0.833 total time=   0.2s
[CV 1/2] END ............model__n_estimators=20;, score=1.000 total time=   0.2s
[CV 2/2] END ............model__n_estimators=20;, score=0.833 total time=   0.2s


GridSearchCV(cv=2,
             estimator=Pipeline(steps=[('feat_scaling', StandardScaler()),
                                       ('feat_selection',
                                        SelectFromModel(estimator=RandomForestClassifier(random_state=101))),
                                       ('model',
                                        RandomForestClassifier(random_state=101))]),
             n_jobs=-2, param_grid={'model__n_estimators': [10, 20]},
             scoring=make_scorer(precision_score, labels=['virginica'], average=None),
             verbose=3)

Next, we check the results for all four different models with `.cv_results_` and use the same code from the previous section
* Note this combination `'model__n_estimators': 10` gave an average precision score on Virginica of 0.91. In this case, both options look to give the same performance, and the grid search picked the model with n_estimator as 10.

In [7]:
(pd.DataFrame(grid.cv_results_)
.sort_values(by='mean_test_score',ascending=False)
.filter(['params','mean_test_score'])
.values
 )

array([[{'model__n_estimators': 10}, 0.9166666666666667],
       [{'model__n_estimators': 20}, 0.9166666666666667]], dtype=object)

We grab programmatically the best hyperparameter combination for a quick check.

In [9]:
grid.best_params_

{'model__n_estimators': 10}

And finally, grab the best pipeline, considering the best cross-validated model for the best hyperparameter combination.

In [10]:
pipeline = grid.best_estimator_
pipeline

Pipeline(steps=[('feat_scaling', StandardScaler()),
                ('feat_selection',
                 SelectFromModel(estimator=RandomForestClassifier(random_state=101))),
                ('model',
                 RandomForestClassifier(n_estimators=10, random_state=101))])

Finally, we evaluate the pipeline.
* Note the precision on Virginica, on the train set, is 98% and on the test set is 100%. It is a very good sign that the precision is maximised for the test set since it shows the pipeline can generalise on unseen data.
* Again, the client will accept the pipeline based on the performance criteria you both set in the ML business case.

In [11]:
from sklearn.metrics import classification_report, confusion_matrix

def confusion_matrix_and_report(X,y,pipeline,label_map):

  prediction = pipeline.predict(X)

  print('---  Confusion Matrix  ---')
  print(pd.DataFrame(confusion_matrix(y_true=prediction, y_pred=y),
        columns=[ ["Actual " + sub for sub in label_map] ], 
        index= [ ["Prediction " + sub for sub in label_map ]]
        ))
  print("\n")


  print('---  Classification Report  ---')
  print(classification_report(y, prediction),"\n")


def clf_performance(X_train,y_train,X_test,y_test,pipeline,label_map):
  print("#### Train Set #### \n")
  confusion_matrix_and_report(X_train,y_train,pipeline,label_map)

  print("#### Test Set ####\n")
  confusion_matrix_and_report(X_test,y_test,pipeline,label_map)
    

clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=pipeline,
                label_map= df_clf['species'].unique()
                )

#### Train Set #### 

---  Confusion Matrix  ---
                      Actual setosa Actual versicolor Actual virginica
Prediction setosa                40                 0                0
Prediction versicolor             0                36                0
Prediction virginica              0                 2               42


---  Classification Report  ---
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        40
  versicolor       1.00      0.95      0.97        38
   virginica       0.95      1.00      0.98        42

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120
 

#### Test Set ####

---  Confusion Matrix  ---
                      Actual setosa Actual versicolor Actual virginica
Prediction setosa                10                 0                0
Prediction versicolor             0                12        

---

<img width="3%" height="3%" align="top"  src="https://codeinstitute.s3.amazonaws.com/predictive_analytics/jupyter_notebook_icons/Icon%209-%20Well%20done.png"> Congratulations! You now know how to get a given algorithm and do a hyperparameter optimisation for Regression and Classification!
  * The **next level** is to define a set of algorithms and a set of hyperparameters for each algorithm and do a hyperparameter optimisation for Regression and Classification tasks!

---